# JuaKazi Gender Bias Engine — Multilingual Training Notebook

**Version:** v2 (Jun 2026)  
**Scope:** All 6 languages — SW, HA, ZU, KI, FR, EN  
**Model:** `Davlan/afro-xlmr-base` — pre-trained on 17 African languages  
**Data:** `training_data_multilingual.csv` (~140K rows, all languages combined)  
**HF target:** `juakazike/multilingual-bias-classifier-v2`  
**Runtime:** Colab T4 GPU — ~90 min  

---

## Why `afro-xlmr-base`?

Pre-trained on Swahili, Hausa, Zulu, Kikuyu, French, English and 11 other African languages. One model replaces 6 separate classifiers. Cross-lingual transfer helps low-resource languages (HA, ZU, KI) borrow signal from the large SW dataset.

## Data prep

Run `scripts/prepare_multilingual_training_data.py` locally, upload `data/training_data_multilingual.csv` to Google Drive at `MyDrive/juakazi/` before running this notebook.


# All Languages — One `afro-xlmr-base` Model


In [ ]:
# ── Cell A1: Install dependencies ─────────────────────────────────────────
import subprocess, sys

def install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

install([
    'transformers>=4.45.0',
    'datasets>=2.18.0',
    'accelerate>=0.34.0',
    'sentencepiece>=0.1.99',
    'scikit-learn>=1.4.0',
    'matplotlib>=3.8.0',
    'seaborn>=0.13.0',
    'huggingface_hub>=0.22.0',
    'protobuf>=4.25.0',
    'nlpaug>=1.1.11',
])
print('✓ Dependencies installed. Please RESTART RUNTIME, then continue from Cell A2.')


In [ ]:
# ── Cell A2: Verify environment ────────────────────────────────────────────
# Run this AFTER restarting runtime
import torch
import transformers
import sklearn

print(f'PyTorch:        {torch.__version__}')
print(f'Transformers:   {transformers.__version__}')
print(f'scikit-learn:   {sklearn.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Switch to T4 GPU runtime before training.')
    raise SystemExit('GPU required')

In [ ]:
# ── Cell A3: Load combined multilingual data (Kaggle) ─────────────────────
import pandas as pd
import numpy as np
import os

# On Kaggle: upload training_data_multilingual.csv as a dataset
# Dataset name: stellaoiro/juakazi-training-data
# It will appear at /kaggle/input/juakazi-training-data/training_data_multilingual.csv
#
# To upload:
#   kaggle.com → Datasets → New Dataset → upload data/training_data_multilingual.csv
#   name it: juakazi-training-data
# Then add it to this notebook: Notebook → + Add Data → search juakazi-training-data

import glob
hits = glob.glob('/kaggle/input/**/training_data_multilingual.csv', recursive=True)
if not hits:
    raise FileNotFoundError('Add juakazi-training-data dataset to this notebook via + Add Data')
INPUT_CSV = hits[0]
DATA_DIR  = os.path.dirname(INPUT_CSV)
print(f'Found data at: {INPUT_CSV}')

print('Loading combined multilingual training data...')
df = pd.read_csv(INPUT_CSV, low_memory=False)

# Drop rows with null text
df = df[df['text'].notna() & (df['text'].str.strip() != '')].copy()

# has_bias: normalise to int
df['has_bias'] = df['has_bias'].astype(str).str.lower().map(
    {'true': 1, '1': 1, 'false': 0, '0': 0}
).fillna(0).astype(int)

# Exclude counter-stereotype — not a bias signal for the classifier
df = df[df['bias_label'] != 'counter-stereotype'].copy()
print(f'  Rows after dropping counter-stereotype: {len(df):,}')

# Held-out EN eval (upload ground_truth_en_v5.csv to the same dataset)
EN_EVAL_PATH = os.path.join(DATA_DIR, 'ground_truth_en_v5.csv')
if os.path.exists(EN_EVAL_PATH):
    en_eval = pd.read_csv(EN_EVAL_PATH, low_memory=False)
    en_eval['has_bias'] = en_eval['has_bias'].astype(str).str.lower().map(
        {'true': 1, '1': 1, 'false': 0, '0': 0}
    ).fillna(0).astype(int)
    print(f'  EN eval: {len(en_eval)} rows')
else:
    en_eval = None
    print('  EN eval: not found (skipping EN eval)')

# Per-language stats
print()
for lang in sorted(df['language'].unique()):
    sub = df[df['language'] == lang]
    biased = sub['has_bias'].sum()
    print(f'  {lang}: {len(sub):>7,} rows | {biased:>6,} biased ({biased/len(sub)*100:.1f}%)')

total_biased = df['has_bias'].sum()
print(f'\n  TOTAL: {len(df):,} rows | {total_biased:,} biased ({total_biased/len(df)*100:.1f}%)')
print('\n✓ Data loaded and verified')


In [ ]:
# ── Cell A4: Config + seeds (Kaggle) ──────────────────────────────────────
import random
import numpy as np
import torch

BASE_MODEL = 'Davlan/afro-xlmr-base'
HF_REPO    = 'juakazike/multilingual-bias-classifier-v2'

SEED         = 42
MAX_LEN      = 128
BATCH_TRAIN  = 32           # per device; Kaggle T4 x2 = effective batch 64
BATCH_EVAL   = 64
EPOCHS       = 5
LR           = 1e-5  # lower LR — HA benefits from slower convergence
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
POS_WEIGHT_CAP = 10.0

# Kaggle T4 x2 supports fp16 reliably — set True for ~2x speed
USE_FP16 = True

FREEZE_LAYERS = 2  # unfreeze more layers for better HA adaptation

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    gpu_count = torch.cuda.device_count()
    print(f'GPUs available: {gpu_count}')
    for i in range(gpu_count):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

print(f'Base model:     {BASE_MODEL}')
print(f'HF target repo: {HF_REPO}')
print(f'Training rows:  {len(df):,}')
print(f'Batch size:     {BATCH_TRAIN} per device (x{torch.cuda.device_count() if torch.cuda.is_available() else 1} GPUs)')
print(f'Epochs:         {EPOCHS}')
print(f'FP16:           {USE_FP16}')
print(f'Frozen layers:  bottom {FREEZE_LAYERS}')
print('✓ Config set')


In [ ]:
# ── Cell A5: Data exploration + class imbalance analysis ────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

datasets_info = [
    ('SW (train+val)', sw,    '#2196F3'),
    ('EN ML train',    en_ml, '#4CAF50'),
    ('FR (train+val)', fr,    '#FF9800'),
]

# Row 0: class distribution pie charts
for i, (name, df, color) in enumerate(datasets_info):
    ax = fig.add_subplot(gs[0, i])
    biased = int(df['has_bias'].sum())
    total  = len(df)
    neutral= total - biased
    ax.pie([neutral, biased], labels=[f'Neutral\n{neutral:,}', f'Biased\n{biased:,}'],
           colors=['#E0E0E0', color], autopct='%1.1f%%', startangle=90)
    ax.set_title(f'{name}\n{total:,} rows', fontsize=11, fontweight='bold')

# Row 1: text length distributions
for i, (name, df, color) in enumerate(datasets_info):
    ax  = fig.add_subplot(gs[1, i])
    lens = df['text'].astype(str).str.split().apply(len)
    ax.hist(lens, bins=50, color=color, alpha=0.7, edgecolor='none')
    ax.axvline(lens.quantile(0.95), color='red', linestyle='--', label=f'p95={lens.quantile(0.95):.0f}')
    ax.set_xlabel('Word count')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name} — text length')
    ax.legend()

plt.suptitle('Dataset Overview — SW / EN / FR', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('/kaggle/working/data_overview.png', dpi=120, bbox_inches='tight')
plt.show()

# Print class weight recommendations
print('\n── Recommended pos_weight values ──')
for name, df, _ in datasets_info:
    n_pos = int(df['has_bias'].sum())
    n_neg = len(df) - n_pos
    raw   = n_neg / n_pos
    capped= min(raw, 10)
    print(f'  {name:20s}: raw={raw:.1f}x  capped={capped:.1f}x')

# Print SW bias categories (what we're trying to detect)
if 'bias_category' in sw.columns:
    print('\n── SW bias categories (biased rows only) ──')
    cats = sw[sw['has_bias']]['bias_category'].value_counts()
    for cat, cnt in cats.items():
        print(f'  {cat:30s}: {cnt:,}')

In [ ]:
# ── Cell A6: Build train/val split from combined dataset ──────────────────
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Use the combined df loaded in A3 — already cleaned, counter-stereotype removed
combined = df.copy()
combined['label'] = combined['has_bias']

# Stratified split — keep language distribution consistent across train/val
train_df, val_df = train_test_split(
    combined, test_size=0.10, random_state=SEED, stratify=combined['language']
)

print(f'Train: {len(train_df):,} rows | Val: {len(val_df):,} rows')
print()
for lang in sorted(combined['language'].unique()):
    tr = train_df[train_df['language'] == lang]
    vl = val_df[val_df['language'] == lang]
    print(f"  {lang}: train={len(tr):,}  val={len(vl):,}  biased_rate={tr['label'].mean():.1%}")

# Class weights for WeightedRandomSampler
n_neg = (train_df['label'] == 0).sum()
n_pos = (train_df['label'] == 1).sum()
raw_weight = n_neg / n_pos
pos_weight  = min(raw_weight, POS_WEIGHT_CAP)
print(f'\nClass weight: {raw_weight:.1f}x (capped at {pos_weight:.1f}x)')

sample_weights = train_df['label'].map({0: 1.0, 1: pos_weight}).values
print('✓ Train/val split ready')


In [ ]:
# ── Cell A7: Tokenizer + length check ──────────────────────────────────────
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

print(f'Loading tokenizer: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Sample 5,000 texts to estimate token length distribution
sample_texts = combined['text'].sample(5000, random_state=SEED).tolist()
token_lengths = [
    len(tokenizer.encode(t, truncation=False)) for t in sample_texts
]

token_lengths = sorted(token_lengths)
p50  = int(np.percentile(token_lengths, 50))
p90  = int(np.percentile(token_lengths, 90))
p95  = int(np.percentile(token_lengths, 95))
p99  = int(np.percentile(token_lengths, 99))
pmax = max(token_lengths)

print(f'Token length percentiles (n=5,000 sample):')
print(f'  p50={p50}  p90={p90}  p95={p95}  p99={p99}  max={pmax}')
print(f'  MAX_LEN={MAX_LEN} covers ~p95 — OK')
pct_truncated = sum(1 for l in token_lengths if l > MAX_LEN) / len(token_lengths)
print(f'  % rows that will be truncated: {pct_truncated*100:.1f}%')

plt.figure(figsize=(10, 4))
plt.hist(token_lengths, bins=80, color='#2196F3', alpha=0.7, edgecolor='none')
plt.axvline(MAX_LEN, color='red', linestyle='--', linewidth=2, label=f'MAX_LEN={MAX_LEN}')
plt.axvline(p95, color='orange', linestyle=':', linewidth=2, label=f'p95={p95}')
plt.xlabel('Token count')
plt.ylabel('Frequency')
plt.title('Token length distribution (5k sample, all languages)')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/token_lengths.png', dpi=120)
plt.show()
print('✓ Tokenizer ready')

In [ ]:
# ── Cell A8: PyTorch Dataset class ──────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class BiasDataset(Dataset):
    """PyTorch dataset for binary gender-bias classification."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Build datasets
train_dataset = BiasDataset(train_df['text'].values, train_df['label'].values, tokenizer, MAX_LEN)
val_dataset   = BiasDataset(val_df['text'].values,   val_df['label'].values,   tokenizer, MAX_LEN)

# Weighted sampler for training (up-samples biased rows)
label_counts = np.bincount(train_df['label'].values)
weights_per_class = 1.0 / label_counts
sample_weights = [weights_per_class[lbl] for lbl in train_df['label'].values]
_sampler_gen = torch.Generator()
_sampler_gen.manual_seed(SEED)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True,
    generator=_sampler_gen,
)

print(f'Train dataset: {len(train_dataset):,} examples')
print(f'Val dataset:   {len(val_dataset):,} examples')
print(f'WeightedRandomSampler: neutral weight={weights_per_class[0]:.6f}, bias weight={weights_per_class[1]:.6f}')
print('Sampler is wired into HuggingFace Trainer via SamplerTrainer._get_train_sampler() (A10/A11).')
print('✓ Datasets ready')

In [ ]:
# ── Cell A9: Load model + freeze bottom layers ──────────────────────────────
import torch
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
model = model.to(device)

# Freeze bottom FREEZE_LAYERS transformer layers to:
#   1. Reduce VRAM usage (critical for T4 16GB)
#   2. Prevent catastrophic forgetting of multilingual representations
#   3. Speed up training by ~25%
frozen_params = 0
total_params  = 0
for name, param in model.named_parameters():
    total_params += param.numel()
    # mDeBERTa layers are named deberta.encoder.layer.{i}.*
    should_freeze = False
    for layer_i in range(FREEZE_LAYERS):
        if f'encoder.layer.{layer_i}.' in name:
            should_freeze = True
            break
    if should_freeze:
        param.requires_grad = False
        frozen_params += param.numel()

trainable_params = total_params - frozen_params
print(f'Total params:     {total_params/1e6:.1f}M')
print(f'Frozen params:    {frozen_params/1e6:.1f}M (bottom {FREEZE_LAYERS} layers)')
print(f'Trainable params: {trainable_params/1e6:.1f}M ({trainable_params/total_params*100:.0f}%)')
print('✓ Model loaded and configured')

In [ ]:
# ── Cell A10: SamplerTrainer + stable eval metrics ───────────────────────────
import torch
import numpy as np
from transformers import Trainer
from transformers.trainer_utils import has_length
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, roc_auc_score,
)


class SamplerTrainer(Trainer):
    """
    Wires WeightedRandomSampler (A8) into HuggingFace Trainer via _get_train_sampler.

    Each training step draws examples with probability ∝ 1 / class_count, so minibatches are
    ~50/50 biased vs neutral in expectation. The model's built-in CrossEntropyLoss (unweighted)
    is then well-conditioned and trains the bias head instead of collapsing to all-neutral.

    Stacking the same sampler with heavy class-weighted CE + fp16 often produced nan eval_loss
    and 0.000 logged train loss on Colab T4.
    """

    def __init__(self, train_balanced_sampler, *args, **kwargs):
        self._train_balanced_sampler = train_balanced_sampler
        super().__init__(*args, **kwargs)

    def _get_train_sampler(self, dataset=None):
        if self.train_dataset is None or not has_length(self.train_dataset):
            return None
        if self.args.group_by_length:
            return super()._get_train_sampler(dataset)
        return self._train_balanced_sampler

    @staticmethod
    def _params_for_grad_norm(model):
        """Trainable params with a grad tensor — excludes frozen encoder layers.
        Passing all `model.parameters()` into Accelerate's `clip_grad_norm_` under fp16
        can trigger: ValueError: Attempting to unscale FP16 gradients (Colab T4 + PyTorch 2.x)."""
        return [p for p in model.parameters() if p.requires_grad and p.grad is not None]

    def _clip_grad_norm(self, model):
        return self.accelerator.clip_grad_norm_(
            self._params_for_grad_norm(model), self.args.max_grad_norm
        )

    def _get_grad_norm(self, model, grad_norm=None):
        if grad_norm is not None:
            return grad_norm.item() if hasattr(grad_norm, "item") else grad_norm
        return self.accelerator.clip_grad_norm_(
            self._params_for_grad_norm(model), float("inf")
        )


def _softmax_stable(logits: np.ndarray) -> np.ndarray:
    x = np.asarray(logits, dtype=np.float64)
    x = x - np.max(x, axis=1, keepdims=True)
    x = np.clip(x, -80.0, 80.0)
    e = np.exp(x)
    return e / np.sum(e, axis=1, keepdims=True)


def compute_metrics(eval_pred):
    """Macro / per-class F1, ROC-AUC (biased = positive class)."""
    logits, labels = eval_pred
    probs = _softmax_stable(logits)[:, 1]
    probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
    preds = (probs >= 0.5).astype(int)
    labels = np.asarray(labels).astype(int)

    report = classification_report(
        labels, preds, target_names=['neutral', 'biased'],
        output_dict=True, zero_division=0,
    )
    try:
        auc = roc_auc_score(labels, probs) if len(np.unique(labels)) >= 2 else 0.0
    except Exception:
        auc = 0.0
    if auc != auc:
        auc = 0.0

    return {
        'f1_macro':        round(f1_score(labels, preds, average='macro', zero_division=0), 4),
        'f1_bias':         round(report['biased']['f1-score'], 4),
        'precision_bias':  round(report['biased']['precision'], 4),
        'recall_bias':     round(report['biased']['recall'], 4),
        'f1_neutral':      round(report['neutral']['f1-score'], 4),
        'roc_auc':         round(float(auc), 4),
        'accuracy':        round(float((preds == labels).mean()), 4),
    }


print('✓ SamplerTrainer and metrics defined')


In [ ]:
# ── Cell A11: TrainingArguments ──────────────────────────────────────────────
from transformers import TrainingArguments
import os

OUTPUT_DIR = '/kaggle/working/multilingual_bias_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

import torch

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_TRAIN,
    per_device_eval_batch_size  = BATCH_EVAL,
    learning_rate               = LR,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = 'cosine',  # cosine decay tends to outperform linear
    eval_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    logging_steps               = 100,
    save_total_limit            = 2,
    fp16                        = torch.cuda.is_available() and USE_FP16,
    max_grad_norm               = 1.0,
    gradient_accumulation_steps = 2,   # effective batch = 32×2 = 64
    dataloader_num_workers      = 2,
    seed                        = SEED,
    report_to                   = 'none',  # disable wandb/tensorboard in Colab
)

trainer = SamplerTrainer(
    train_balanced_sampler = sampler,
    model                  = model,
    args                   = training_args,
    train_dataset          = train_dataset,
    eval_dataset           = val_dataset,
    compute_metrics        = compute_metrics,
)

# Estimate training time
steps_per_epoch = len(train_dataset) // (BATCH_TRAIN * training_args.gradient_accumulation_steps)
total_steps     = steps_per_epoch * EPOCHS
print(f'Steps per epoch:  {steps_per_epoch:,}')
print(f'Total steps:      {total_steps:,}')
print(f'Effective batch:  {BATCH_TRAIN * training_args.gradient_accumulation_steps}')
print(f'Estimated time:   ~{total_steps * 2.5 / 60:.0f} min on T4')
print('✓ Trainer configured — ready to train')

In [ ]:
# ── Cell A12: TRAIN ──────────────────────────────────────────────────────────
# ⚠️  This cell takes ~60-90 minutes on T4 GPU
# Progress logs every 100 steps below

print('Starting training...')
print(f'Model:   {BASE_MODEL}')
print(f'Epochs:  {EPOCHS}')
print(f'Device:  {device}')
print('-' * 60)

result = trainer.train()

print('-' * 60)
print(f'Training complete.')
print(f'  Total steps:         {result.global_step:,}')
print(f'  Training loss:       {result.training_loss:.4f}')
print(f'  Training time:       {result.metrics["train_runtime"]:.0f}s ({result.metrics["train_runtime"]/60:.1f} min)')
print(f'  Samples/sec:         {result.metrics["train_samples_per_second"]:.1f}')

In [ ]:
# ── Cell A13: Loss + F1 training curves ──────────────────────────────────────
import matplotlib.pyplot as plt

# Extract per-epoch metrics from trainer state
history = trainer.state.log_history

epochs_data     = []
train_loss_data = []
eval_f1_data    = []
eval_prec_data  = []
eval_rec_data   = []

for entry in history:
    if 'eval_f1_macro' in entry:
        epochs_data.append(entry.get('epoch', 0))
        eval_f1_data.append(entry['eval_f1_macro'])
        eval_prec_data.append(entry.get('eval_precision_bias', 0))
        eval_rec_data.append(entry.get('eval_recall_bias', 0))
    if 'loss' in entry and 'eval_loss' not in entry:
        train_loss_data.append(entry['loss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(range(len(train_loss_data)), train_loss_data, color='#F44336', linewidth=1.5)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Training loss')
axes[0].set_title('Training loss')
axes[0].grid(alpha=0.3)

# F1/P/R curves
axes[1].plot(epochs_data, eval_f1_data,    label='Macro F1',      marker='o', color='#2196F3')
axes[1].plot(epochs_data, eval_prec_data,  label='Bias Precision', marker='s', color='#4CAF50', linestyle='--')
axes[1].plot(epochs_data, eval_rec_data,   label='Bias Recall',    marker='^', color='#FF9800', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Validation metrics')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=120)
plt.show()

print('Best model checkpoint loaded (load_best_model_at_end=True)')

In [ ]:
# ── Cell A13b: PERFORMANCE GATE — v2 must beat v1 before pushing ────────────
# v1 baselines (eval/metrics.json as of 2026-06-17, lexicon+ML combined)
# v2 MUST exceed ALL of these or training is rejected.
V1_BASELINES = {
    'sw': {'f1': 0.850, 'precision': 0.823, 'recall': 0.880},
    'ha': {'f1': 0.382, 'precision': 0.425, 'recall': 0.347},
    'zu': {'f1': 0.965, 'precision': 0.932, 'recall': 1.000},
}

# Map eval result names to lang codes
RESULT_LANG_MAP = {
    'SW (val)': 'sw',
    'EN ML (val)': 'en',
    'FR (val)': 'fr',
}

print('=' * 60)
print('PERFORMANCE GATE — v2 vs v1 baselines')
print('=' * 60)

gate_passed = True
for result_name, lang in RESULT_LANG_MAP.items():
    if lang not in V1_BASELINES:
        continue
    if result_name not in results:
        print(f'  {lang.upper()}: SKIP (no eval results)')
        continue
    r = results[result_name]
    baseline = V1_BASELINES[lang]
    f1_ok   = r['f1_macro']  > baseline['f1']
    rec_ok  = r['recall']    >= baseline['recall'] - 0.02  # 2pp tolerance on recall
    prec_ok = r['precision'] >= baseline['precision'] - 0.05  # 5pp tolerance on precision
    ok = f1_ok
    status = '✅ PASS' if ok else '❌ FAIL'
    print(f"  {lang.upper()}: F1={r['f1_macro']:.3f} (v1={baseline['f1']:.3f}) | "          f"P={r['precision']:.3f} (v1={baseline['precision']:.3f}) | "          f"R={r['recall']:.3f} (v1={baseline['recall']:.3f})  {status}")
    if not ok:
        gate_passed = False

print()
if gate_passed:
    print('✅ ALL GATES PASSED — safe to push to HuggingFace')
else:
    print('❌ GATE FAILED — DO NOT PUSH. Investigate before proceeding.')
    print('   Options:')
    print('   1. Increase EPOCHS (try 6-7)')
    print('   2. Lower LR further (try 5e-6)')
    print('   3. Reduce FREEZE_LAYERS to 0')
    print('   4. Check class balance in training data')
    raise RuntimeError('Performance gate failed — v2 does not outperform v1. Do not push.')

print('=' * 60)


In [ ]:
# ── Cell A14: Full evaluation — per-language breakdown ───────────────────────
import torch
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
device = next(model.parameters()).device

def evaluate_df(df, lang_label, threshold=0.5):
    """Run model on a dataframe, return dict of metrics."""
    dataset = BiasDataset(df['text'].values, df['label'].values, tokenizer, MAX_LEN)
    loader  = DataLoader(dataset, batch_size=BATCH_EVAL, shuffle=False)

    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_batch   = batch['labels'].numpy()

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()

            all_probs.extend(probs)
            all_labels.extend(labels_batch)

    preds = (np.array(all_probs) >= threshold).astype(int)
    labels_arr = np.array(all_labels)

    return {
        'lang':      lang_label,
        'n':         len(labels_arr),
        'f1_macro':  f1_score(labels_arr, preds, average='macro', zero_division=0),
        'precision': precision_score(labels_arr, preds, zero_division=0),
        'recall':    recall_score(labels_arr, preds, zero_division=0),
        'probs':     all_probs,
        'labels':    all_labels,
    }

# Evaluate on each language separately
# Build per-language val sets from val_df
val_sw = val_df[val_df['language'] == 'sw'].copy()  # keeps all cols including target_gender, bias_category
val_en = val_df[val_df['language'] == 'en'].copy()
val_fr = val_df[val_df['language'] == 'fr'].copy()

# EN eval holdout (never seen during training)
if en_eval is not None:
    en_eval_set = en_eval.copy()
    en_eval_set['label'] = en_eval_set['has_bias'].astype(int)
else:
    en_eval_set = None
    print('EN holdout eval CSV not found — skipping EN holdout evaluation')

print('Running evaluation...')
results = {}
base_pairs = [('SW (val)', val_sw), ('EN ML (val)', val_en), ('FR (val)', val_fr)]
en_eval_pairs = [('EN holdout', en_eval_set)] if en_eval_set is not None else []
for name, df in base_pairs + en_eval_pairs:
    if df is None or len(df) == 0:
        print(f'  {name}: skipped (0 rows)')
        continue
    r = evaluate_df(df, name)
    results[name] = r
    print(f'  {name:22s}: F1={r["f1_macro"]:.3f}  P={r["precision"]:.3f}  R={r["recall"]:.3f}  (n={r["n"]})')

print('\n✓ Evaluation complete')

In [ ]:
# ── Cell A15: Gender-disaggregated metrics ───────────────────────────────────
# Evaluates whether the model performs equally well on male vs female bias
# AIBRIDGE specifically requires this disaggregation

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

print('=== Gender-disaggregated evaluation (SW val set) ===')
print('Requires target_gender column in SW data\n')

if 'target_gender' in val_sw.columns:
    # Merge val_sw with original SW to get target_gender
    val_sw_meta = val_df[val_df['language'] == 'sw'].copy()
    # Get probabilities from the evaluation we already ran
    sw_result = results.get('SW (val)', None)

    if sw_result:
        probs  = np.array(sw_result['probs'])
        labels = np.array(sw_result['labels'])
        preds  = (probs >= 0.5).astype(int)

        # Look up target_gender from the val split
        val_sw_reset = val_sw.reset_index(drop=True)

        for gender in ['female', 'male']:
            mask = (val_sw_reset['target_gender'] == gender).values
            if mask.sum() == 0:
                continue
            g_labels = labels[mask]
            g_preds  = preds[mask]
            biased_mask = g_labels == 1
            if biased_mask.sum() == 0:
                continue
            f1 = f1_score(g_labels, g_preds, average='macro', zero_division=0)
            p  = precision_score(g_labels, g_preds, zero_division=0)
            r  = recall_score(g_labels, g_preds, zero_division=0)
            print(f'  {gender:8s}: n_biased={biased_mask.sum():3d}  F1={f1:.3f}  P={p:.3f}  R={r:.3f}')

        # Gap analysis
        print('\nNote: Goal is female recall within 5pp of male recall (AIBRIDGE fairness criterion)')
else:
    print('  target_gender column not present in val split — run full SW eval with metadata')
    print('  Skipping disaggregated eval for now.')

# Also print bias_category breakdown if available
if 'bias_category' in val_sw.columns:
    print('\n=== Bias category breakdown (SW val, biased rows only) ===')
    sw_result = results.get('SW (val)', None)
    if sw_result:
        probs  = np.array(sw_result['probs'])
        labels = np.array(sw_result['labels'])
        preds  = (probs >= 0.5).astype(int)
        val_sw_reset = val_sw.reset_index(drop=True)

        # Only look at biased rows (label=1)
        biased_mask = labels == 1
        for cat in val_sw_reset.loc[biased_mask, 'bias_category'].value_counts().index[:8]:
            cat_mask = (val_sw_reset['bias_category'] == cat).values & biased_mask
            if cat_mask.sum() < 3:
                continue
            cat_recall = (preds[cat_mask] == labels[cat_mask]).mean()
            print(f'  {cat:30s}: n={cat_mask.sum():3d}  recall={cat_recall:.2f}')

In [ ]:
# ── Cell A16: Per-language optimal threshold tuning ─────────────────────────
# Different languages may have different optimal decision thresholds
# because their bias rate and text style differ

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score

THRESHOLDS = np.arange(0.10, 0.95, 0.01)

optimal_thresholds = {}

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4), squeeze=False)

for idx, (name, result) in enumerate(results.items()):
    probs  = np.array(result['probs'])
    labels = np.array(result['labels'])

    f1s    = []
    precs  = []
    recs   = []

    for t in THRESHOLDS:
        preds = (probs >= t).astype(int)
        f1s.append(f1_score(labels, preds, average='macro', zero_division=0))
        precs.append(precision_score(labels, preds, zero_division=0))
        recs.append(recall_score(labels, preds, zero_division=0))

    best_t   = THRESHOLDS[np.argmax(f1s)]
    best_f1  = max(f1s)
    best_idx = np.argmax(f1s)
    optimal_thresholds[name] = float(best_t)

    ax = axes[0][idx]
    ax.plot(THRESHOLDS, f1s,   label='Macro F1',  color='#2196F3')
    ax.plot(THRESHOLDS, precs, label='Precision', color='#4CAF50', linestyle='--')
    ax.plot(THRESHOLDS, recs,  label='Recall',    color='#FF9800', linestyle='--')
    ax.axvline(best_t, color='red', linestyle=':', linewidth=2, label=f'best t={best_t:.2f}')
    ax.set_title(f'{name}\nbest F1={best_f1:.3f} @ t={best_t:.2f}', fontsize=10)
    ax.set_xlabel('Threshold')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    print(f'  {name}: optimal threshold={best_t:.2f}  (F1={best_f1:.3f}  P={precs[best_idx]:.3f}  R={recs[best_idx]:.3f})')

plt.tight_layout()
plt.savefig('/kaggle/working/threshold_tuning.png', dpi=120)
plt.show()

print('\nOptimal thresholds dict (save these — embed in model config.json):')
print(optimal_thresholds)

In [ ]:
# ── Cell A17: Save model + metadata + push to HuggingFace Hub ───────────────
import json
import os
import numpy as np
from datetime import datetime
from huggingface_hub import HfApi, login
import os

# Get HF token from Kaggle secrets:
# Notebook → Add-ons → Secrets → add HF_TOKEN
# (or set it as an env var in Kaggle notebook settings)
HF_TOKEN = os.environ.get('HF_TOKEN') or input('Paste your HuggingFace write token: ').strip()

login(token=HF_TOKEN)

SAVE_DIR = '/kaggle/working/multilingual_bias_v2_final'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model + tokenizer
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save metadata JSON (used by the inference API to load thresholds)
metadata = {
    'model_id':           HF_REPO,
    'base_model':         BASE_MODEL,
    'version':            'v2',
    'task':               'gender-bias-detection',
    'languages':          ['sw', 'ha', 'zu', 'ki', 'fr', 'en'],
    'trained_on':         datetime.utcnow().strftime('%Y-%m-%d'),
    'training_samples':   len(train_df),
    'val_samples':        len(val_df),
    'max_length':         MAX_LEN,
    'optimal_thresholds': optimal_thresholds,
    'pos_weight':         float((train_df['label'] == 0).sum() / max((train_df['label'] == 1).sum(), 1)),
    'freeze_layers':      FREEZE_LAYERS,
    'final_metrics':      {
        name: {
            'f1_macro':  round(r['f1_macro'],  4),
            'precision': round(r['precision'], 4),
            'recall':    round(r['recall'],    4),
        }
        for name, r in results.items()
    }
}

with open(f'{SAVE_DIR}/juakazi_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved metadata:')
print(json.dumps(metadata, indent=2))

# Push to HuggingFace Hub
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True, private=False)

api.upload_folder(
    folder_path    = SAVE_DIR,
    repo_id        = HF_REPO,
    repo_type      = 'model',
    commit_message = f'Train multilingual-bias-classifier-v2 ({datetime.utcnow().strftime("%Y-%m-%d")})',
)

print(f'\n✓ Model pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── Cell A18: Integration test — verify model loads and predicts correctly ────
import torch
from transformers import pipeline

print(f'Loading model from HF Hub: {HF_REPO}')
pipe = pipeline(
    'text-classification',
    model=HF_REPO,
    tokenizer=HF_REPO,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=MAX_LEN,
)

test_cases = [
    # (text, lang, expected_label, description)
    ('The doctor called his patient.',          'en', 'LABEL_0', 'EN: gendered pronoun — neutral context'),
    ('The nurse called her patient.',           'en', 'LABEL_0', 'EN: gendered pronoun — neutral context'),
    ('Women are too emotional to lead.',        'en', 'LABEL_1', 'EN: explicit occupational bias'),
    ('Wanawake wanafaa kuwa nyumbani tu.',      'sw', 'LABEL_1', 'SW: women should stay home — biased'),
    ('Daktari alifanya kazi nzuri leo.',        'sw', 'LABEL_0', 'SW: neutral medical statement'),
    ('Les femmes sont trop émotives pour gérer.', 'fr', 'LABEL_1', 'FR: explicit bias'),
    ('La chercheuse a publié ses résultats.',   'fr', 'LABEL_0', 'FR: female researcher — neutral'),
]

print('\nIntegration test results:')
print(f'{"Text"[:50]:50s}  {"Expected":10s}  {"Got":10s}  {"Score":8s}  Pass?')
print('-' * 90)
all_pass = True
for text, lang, expected, desc in test_cases:
    result = pipe(text)[0]
    got    = result['label']
    score  = result['score']
    passed = got == expected
    if not passed:
        all_pass = False
    status = '✓' if passed else '✗'
    print(f'{desc[:50]:50s}  {expected:10s}  {got:10s}  {score:.4f}   {status}')

print(f'\n{"All tests passed ✓" if all_pass else "Some tests failed ✗ — review thresholds or training"}')

---

# PART B — Kikuyu Model (`Davlan/afro-xlmr-large` + DAPT)

## Why a separate KI model?

The joint SW/EN/FR model cannot be used for KI because:
1. **KI recall is 0.256** — fundamentally broken, not fixable by threshold tuning
2. **79% of KI biased rows are implicit bias** — rules don't catch them
3. **Kikuyu-specific terminology** — the model needs exposure to Gikuyu morphology

## DAPT strategy

Domain-Adaptive Pre-Training (DAPT) masks 15% of tokens from raw Gikuyu text and runs MLM
for 1-2 epochs on `AfroXLMR-large` before fine-tuning. This adapts the model's internal
representations to Gikuyu vocabulary without full pre-training from scratch.

**DAPT corpus:** Gikuyu Wikipedia (~50K sentences) + Kikuyu Bible (NT + OT, ~30K sentences)

**Expected improvement:** AfroXLMR-Social paper reports 1–30% F1 improvement from DAPT on
same-language corpora. For KI we target F1 0.401 → 0.55+.


In [ ]:
# ── Cell B1: Load KI data + baseline analysis ────────────────────────────────
# ⚠️  RESTART RUNTIME before running Part B to free VRAM

import pandas as pd
import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split

# Settings
KI_BASE_MODEL  = 'Davlan/afro-xlmr-large'
KI_HF_REPO     = 'juakazike/ki-bias-classifier-v1'
SEED           = 42
MAX_LEN_KI     = 128
BATCH_TRAIN_KI = 16   # afro-xlmr-large is larger — reduce batch size
BATCH_EVAL_KI  = 32
EPOCHS_KI      = 5
LR_KI          = 1e-5  # lower LR for larger model
DAPT_EPOCHS    = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kaggle: data already loaded from dataset added in A3

DRIVE = '/kaggle/input/juakazi-training-data'

ki = pd.read_csv(f'{DRIVE}/ground_truth_ki_v8.csv', low_memory=False)

# Validate
assert ki['has_bias'].isna().sum() == 0, 'NaN has_bias in KI'
assert ki['text'].isna().sum() == 0, 'null texts in KI'
biased = int(ki['has_bias'].sum())
total  = len(ki)
print(f'KI dataset: {total:,} rows | {biased:,} biased ({biased/total*100:.1f}%)')

# Bias category breakdown
if 'bias_category' in ki.columns:
    print('\nBias categories (biased rows):')
    cats = ki[ki['has_bias']]['bias_category'].value_counts()
    for cat, cnt in cats.items():
        print(f'  {cat:30s}: {cnt:,} ({cnt/biased*100:.0f}%)')

if 'explicitness' in ki.columns:
    print('\nExplicitness breakdown:')
    exp = ki[ki['has_bias']]['explicitness'].value_counts()
    for e, cnt in exp.items():
        print(f'  {e:15s}: {cnt:,} ({cnt/biased*100:.0f}%)')

print('\n✓ KI data loaded')

In [ ]:
# ── Cell B2: DAPT — Domain-Adaptive Pre-Training on Gikuyu corpus ────────────
#
# What DAPT does:
#   1. Load AfroXLMR-large (already has Gikuyu in pre-training)
#   2. Fine-tune the MLM head on raw Gikuyu text (Masked Language Modeling)
#   3. This adapts representations to Gikuyu domain vocabulary
#   4. Save the DAPT-adapted model — use it as base for fine-tuning in B3
#
# Input: raw Gikuyu text (no labels needed — self-supervised)
# If you have the Gikuyu Wikipedia dump, place it at:
#   MyDrive/juakazi/ki_dapt_corpus.txt  (one sentence per line)
# If not available, Cell B2 will be skipped and B3 will use AfroXLMR directly.

import os
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)
from datasets import Dataset

DAPT_CORPUS = f'{DRIVE}/ki_dapt_corpus.txt'
DAPT_DIR    = '/kaggle/working/afro_xlmr_ki_dapt'

if not os.path.exists(DAPT_CORPUS):
    print(f'DAPT corpus not found at {DAPT_CORPUS}')
    print('Skipping DAPT — will fine-tune directly on AfroXLMR-large.')
    print('To enable DAPT:')
    print('  1. Download Gikuyu Wikipedia: https://dumps.wikimedia.org/kiwiki/')
    print('  2. Extract sentences using WikiExtractor')
    print('  3. Save as ki_dapt_corpus.txt in MyDrive/juakazi/')
    DAPT_DIR = KI_BASE_MODEL  # fall back to original model
else:
    print(f'DAPT corpus found. Loading...')
    with open(DAPT_CORPUS) as f:
        lines = [l.strip() for l in f if len(l.strip()) > 20]
    print(f'  {len(lines):,} sentences for DAPT')

    tokenizer_dapt = AutoTokenizer.from_pretrained(KI_BASE_MODEL)
    model_dapt     = AutoModelForMaskedLM.from_pretrained(KI_BASE_MODEL)

    # Tokenize corpus
    def tokenize_fn(batch):
        return tokenizer_dapt(batch['text'], truncation=True, max_length=128, padding=False)

    raw_ds = Dataset.from_dict({'text': lines})
    tok_ds = raw_ds.map(tokenize_fn, batched=True, remove_columns=['text'])

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer_dapt, mlm=True, mlm_probability=0.15
    )

    os.makedirs(DAPT_DIR, exist_ok=True)
    dapt_args = TrainingArguments(
        output_dir              = DAPT_DIR,
        num_train_epochs        = DAPT_EPOCHS,
        per_device_train_batch_size = 32,
        learning_rate           = 5e-5,
        warmup_ratio            = 0.1,
        weight_decay            = 0.01,
        save_strategy           = 'epoch',
        load_best_model_at_end  = False,
        logging_steps           = 200,
        fp16                    = False,  # disable fp16 in DAPT — GradScaler issues on Colab T4
        report_to               = 'none',
    )

    dapt_trainer = Trainer(
        model          = model_dapt,
        args           = dapt_args,
        train_dataset  = tok_ds,
        data_collator  = data_collator,
    )

    print(f'Starting DAPT ({DAPT_EPOCHS} epochs on {len(tok_ds):,} examples)...')
    dapt_trainer.train()

    # Save DAPT model (will be used as base in B3)
    model_dapt.save_pretrained(DAPT_DIR)
    tokenizer_dapt.save_pretrained(DAPT_DIR)
    print(f'✓ DAPT complete. Adapted model saved to {DAPT_DIR}')

print(f'\nKI fine-tuning will use base model: {DAPT_DIR}')

In [ ]:
# ── Cell B3: Fine-tune KI bias classifier ────────────────────────────────────
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from transformers.trainer_utils import has_length
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load tokenizer from DAPT output (or original if DAPT was skipped)
tokenizer_ki = AutoTokenizer.from_pretrained(DAPT_DIR)

# Prepare KI data
ki_clean = ki[ki['bias_label'] != 'annotation_error'].copy()
ki_clean['label'] = ki_clean['has_bias'].astype(int)

# Sampling: KI is 11.3% biased — use 10:1 neutral:biased ratio
ki_biased  = ki_clean[ki_clean['label'] == 1]
ki_neutral = ki_clean[ki_clean['label'] == 0].sample(
    n=min(len(ki_biased) * 10, len(ki_clean[ki_clean['label']==0])),
    random_state=SEED
)
ki_sampled = pd.concat([ki_biased, ki_neutral]).sample(frac=1, random_state=SEED).reset_index(drop=True)

ki_train_df, ki_val_df = train_test_split(
    ki_sampled, test_size=0.15, random_state=SEED, stratify=ki_sampled['label']
)

print(f'KI train: {len(ki_train_df):,} rows (biased: {ki_train_df["label"].sum():,})')
print(f'KI val:   {len(ki_val_df):,} rows (biased: {ki_val_df["label"].sum():,})')

# Dataset class (reuse from Part A)
class BiasDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_len,
                              padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

ki_train_ds = BiasDataset(ki_train_df['text'].values, ki_train_df['label'].values, tokenizer_ki, MAX_LEN_KI)
ki_val_ds   = BiasDataset(ki_val_df['text'].values,   ki_val_df['label'].values,   tokenizer_ki, MAX_LEN_KI)

# Balanced training batches (same pattern as Part A — do not create a sampler and omit it from Trainer)
_ki_counts = np.bincount(ki_train_df['label'].values)
_ki_wpc = 1.0 / _ki_counts
_ki_sample_w = [_ki_wpc[lbl] for lbl in ki_train_df['label'].values]
_ki_gen = torch.Generator().manual_seed(SEED)
ki_sampler = WeightedRandomSampler(
    weights=_ki_sample_w, num_samples=len(ki_train_ds), replacement=True, generator=_ki_gen,
)
print(f'KI WeightedRandomSampler: w_neutral={_ki_wpc[0]:.6f}, w_bias={_ki_wpc[1]:.6f}')

# Model
model_ki = AutoModelForSequenceClassification.from_pretrained(
    DAPT_DIR, num_labels=2, ignore_mismatched_sizes=True
).to(device)

# Metadata only (Hub card / debugging) — training uses ki_sampler + unweighted CE
n_pos_ki  = int(ki_train_df['label'].sum())
n_neg_ki  = len(ki_train_df) - n_pos_ki
pos_weight_ki = float(min(n_neg_ki / max(n_pos_ki, 1), 8.0))
print(f'KI pos_weight (metadata): {pos_weight_ki:.1f}x (raw: {n_neg_ki/max(n_pos_ki,1):.1f}x)')


def _ki_softmax(logits):
    x = np.asarray(logits, dtype=np.float64)
    x = x - np.max(x, axis=1, keepdims=True)
    x = np.clip(x, -80.0, 80.0)
    e = np.exp(x)
    return e / np.sum(e, axis=1, keepdims=True)


def compute_metrics_ki(eval_pred):
    logits, labels = eval_pred
    probs = _ki_softmax(logits)[:, 1]
    probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
    preds = (probs >= 0.5).astype(int)
    labels = np.asarray(labels).astype(int)
    rep = classification_report(labels, preds, target_names=['neutral', 'biased'], output_dict=True, zero_division=0)
    return {
        'f1_macro':        round(f1_score(labels, preds, average='macro', zero_division=0), 4),
        'f1_bias':         round(rep['biased']['f1-score'], 4),
        'precision_bias':  round(rep['biased']['precision'], 4),
        'recall_bias':     round(rep['biased']['recall'], 4),
    }


class SamplerTrainerKI(Trainer):
    """Same as Part A SamplerTrainer — required after Part B runtime restart."""

    def __init__(self, train_balanced_sampler, *args, **kwargs):
        self._train_balanced_sampler = train_balanced_sampler
        super().__init__(*args, **kwargs)

    def _get_train_sampler(self, dataset=None):
        if self.train_dataset is None or not has_length(self.train_dataset):
            return None
        if self.args.group_by_length:
            return super()._get_train_sampler(dataset)
        return self._train_balanced_sampler

    @staticmethod
    def _params_for_grad_norm(model):
        return [p for p in model.parameters() if p.requires_grad and p.grad is not None]

    def _clip_grad_norm(self, model):
        return self.accelerator.clip_grad_norm_(
            self._params_for_grad_norm(model), self.args.max_grad_norm
        )

    def _get_grad_norm(self, model, grad_norm=None):
        if grad_norm is not None:
            return grad_norm.item() if hasattr(grad_norm, "item") else grad_norm
        return self.accelerator.clip_grad_norm_(
            self._params_for_grad_norm(model), float("inf")
        )


KI_USE_FP16 = False  # match Part A: fp16 on Colab often breaks GradScaler + grad norm

ki_args = TrainingArguments(
    output_dir                  = '/kaggle/working/ki_bias_v1',
    num_train_epochs            = EPOCHS_KI,
    per_device_train_batch_size = BATCH_TRAIN_KI,
    per_device_eval_batch_size  = BATCH_EVAL_KI,
    learning_rate               = LR_KI,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_bias',
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available() and KI_USE_FP16,
    max_grad_norm               = 1.0,
    gradient_accumulation_steps = 4,  # effective batch = 64
    logging_steps               = 50,
    save_total_limit            = 2,
    report_to                   = 'none',
    seed                        = SEED,
)

ki_trainer = SamplerTrainerKI(
    train_balanced_sampler = ki_sampler,
    model                  = model_ki,
    args                   = ki_args,
    train_dataset          = ki_train_ds,
    eval_dataset           = ki_val_ds,
    compute_metrics        = compute_metrics_ki,
)

print('Starting KI training...')
ki_result = ki_trainer.train()
print(f'\nKI training complete. Steps: {ki_result.global_step:,}')


In [ ]:
# ── Cell B4: KI evaluation + threshold tuning ────────────────────────────────
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import matplotlib.pyplot as plt

model_ki.eval()
device = next(model_ki.parameters()).device

# Full KI val set
ki_val_all_ds = BiasDataset(ki_val_df['text'].values, ki_val_df['label'].values, tokenizer_ki, MAX_LEN_KI)
loader = DataLoader(ki_val_all_ds, batch_size=BATCH_EVAL_KI, shuffle=False)

all_probs, all_labels = [], []
with torch.no_grad():
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].numpy()
        outs = model_ki(input_ids=ids, attention_mask=mask)
        probs = torch.softmax(outs.logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(lbls)

probs_arr  = np.array(all_probs)
labels_arr = np.array(all_labels)

# Find optimal KI threshold
thresholds = np.arange(0.10, 0.95, 0.01)
f1s = [f1_score(labels_arr, (probs_arr >= t).astype(int), average='macro', zero_division=0) for t in thresholds]
best_t_ki  = thresholds[np.argmax(f1s)]
best_preds = (probs_arr >= best_t_ki).astype(int)

print('=== KI Final Evaluation ===')
print(f'Optimal threshold: {best_t_ki:.2f}')
print(classification_report(labels_arr, best_preds, target_names=['neutral', 'biased'], zero_division=0))

# Plot threshold curve
plt.figure(figsize=(10, 4))
plt.plot(thresholds, f1s, color='#2196F3', label='Macro F1')
plt.axvline(best_t_ki, color='red', linestyle='--', label=f'best t={best_t_ki:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Macro F1')
plt.title('KI threshold tuning')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/ki_threshold.png', dpi=120)
plt.show()

ki_metrics = {
    'f1_macro':  round(f1_score(labels_arr, best_preds, average='macro', zero_division=0), 4),
    'precision': round(precision_score(labels_arr, best_preds, zero_division=0), 4),
    'recall':    round(recall_score(labels_arr, best_preds, zero_division=0), 4),
    'threshold': float(best_t_ki),
}
print(f'\nKI metrics @ t={best_t_ki:.2f}:', ki_metrics)

In [ ]:
# ── Cell B5: Save KI model + push to HuggingFace Hub ────────────────────────
import json
import os
from datetime import datetime
from huggingface_hub import HfApi, login
import os
HF_TOKEN = os.environ.get('HF_TOKEN') or input('HuggingFace write token: ').strip()

login(token=HF_TOKEN)

KI_SAVE_DIR = '/kaggle/working/ki_bias_final'
os.makedirs(KI_SAVE_DIR, exist_ok=True)

model_ki.save_pretrained(KI_SAVE_DIR)
tokenizer_ki.save_pretrained(KI_SAVE_DIR)

ki_metadata = {
    'model_id':           KI_HF_REPO,
    'base_model':         KI_BASE_MODEL,
    'dapt_applied':       DAPT_DIR != KI_BASE_MODEL,
    'dapt_base':          DAPT_DIR,
    'version':            'v1',
    'task':               'gender-bias-detection',
    'languages':          ['ki'],
    'trained_on':         datetime.utcnow().strftime('%Y-%m-%d'),
    'training_samples':   len(ki_train_df),
    'val_samples':        len(ki_val_df),
    'max_length':         MAX_LEN_KI,
    'optimal_threshold':  ki_metrics['threshold'],
    'pos_weight':         float(pos_weight_ki),
    'val_metrics':        ki_metrics,
}

with open(f'{KI_SAVE_DIR}/juakazi_metadata.json', 'w') as f:
    json.dump(ki_metadata, f, indent=2)

api = HfApi()
api.create_repo(KI_HF_REPO, exist_ok=True, private=False)
api.upload_folder(
    folder_path    = KI_SAVE_DIR,
    repo_id        = KI_HF_REPO,
    repo_type      = 'model',
    commit_message = f'Train ki-bias-classifier-v1 ({datetime.utcnow().strftime("%Y-%m-%d")})',
)

print(f'\n✓ KI model pushed to: https://huggingface.co/{KI_HF_REPO}')
print(f'\n=== TRAINING COMPLETE ===')
print(f'SW/EN/FR model: https://huggingface.co/juakazike/multilingual-bias-classifier-v1')
print(f'KI model:       https://huggingface.co/{KI_HF_REPO}')
print(f'\nNext steps:')
print(f'  1. Update eval/ml_classifier.py: _MODEL_ID = "juakazike/multilingual-bias-classifier-v1"')
print(f'  2. Add KI model routing in api/service.py for language="ki"')
print(f'  3. Run: python3 run_evaluation.py to confirm F1 improvements')
print(f'  4. Run: python3 tests/test_system.py — must stay 5/5')
print(f'  5. Update CLAUDE.md metrics table with new scores')